## Gradient Descent - Backpropagation
Gradient descent is an optimization algorithm used to minimize a loss function by iteratively adjusting the parameters of a model. In the context of neural networks, backpropagation is the method used to compute the gradients of the loss function with respect to the model's parameters.

**Types:**
- Batch Gradient Descent:
    - Uses the entire dataset to compute the gradient at each step.
- Stochastic Gradient Descent (SGD): Uses a single data point to compute the gradient at each step.
    - Uses a single data point to compute the gradient at each step.
- Mini-batch Gradient Descent:
    - Uses a small subset of the dataset to compute the gradient at each step.

**How it works?**
So the goal is to adjust the weights/parameters of the model such that the loss function is minimized or in other terms the predictions of the model are as close to the actual target values as possible. So how we do adjust these weights/parameters? This is where gradient descent comes into play.
1. **Initialization**: Start with random values for the weights/parameters of the model.
2. **Forward Pass**: Pass the input data through the model to obtain predictions.
3. **Compute Loss**: Calculate the loss using a loss function (e.g., Mean Squared Error, Cross-Entropy Loss) that quantifies the difference between the predicted values and the actual target values. Here the loss function depends on the type of problem we are solving. For regression problems, Mean Squared Error (MSE) is commonly used, while for classification problems, Cross-Entropy Loss is often preferred.
4. **Backward Pass (Backpropagation)**: Compute the gradients (partial derivatives) of the loss function with respect to each weight/parameter in the model using the chain rule of calculus. This step involves propagating the error backward through the network.
5. **Update Weights**: Adjust the weights/parameters using the computed gradients and a learning rate. The learning rate determines the step size at each iteration while moving toward a minimum of the loss function. The learning rate should be chosen carefully; a value too high can cause the model to overshoot on the global minimum or diverge, while a value too low can result in slow convergence.

    - $param_{new} = param_{old} + learning\_rate * (-gradient)$
    - $gradient = \frac{∂loss}{∂param}$

- Let's implement it in PyTorch (manually):
    - we will assume 2 layers neural network with ReLU activation function.

- Forward Pass:
    - $z_1 = X * W_1 + b_1$
    - $a_1 = ReLU(z_1)$
    - $z_2 = a_1 * W_2 + b_2$
    - $y\_pred = z_2$

- Backward Pass:
    - Compute loss: $L = CrossEntropy(z_2, y\_true)$
    - Compute gradients using chain rule:
        - $dL/dz_2 = softmax(z_2) - y\_true$
        - $dL/dW_2 = a_1^T @ dL/dz_2$
        - $dL/db_2 = sum(dL/dz_2)$
        - $dL/da_1 = dL/dz_2 * W_2^T$
        - $dL/dz_1 = dL/da_1 * ReLU'(z_1)$
        - $dL/dW_1 = X^T * dL/dz_1$
        - $dL/db_1 = sum(dL/dz_1)$

- So lets use the FFN example from our GPT implementation and manually implement the backpropagation using gradient descent.

In [7]:
import torch
import torch.nn as nn

class GELU(nn.Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, x):
        # tanh approximation
        k = torch.sqrt(torch.tensor(2.0/torch.pi))
        return 0.5 * x * (1 + torch.tanh(k * (x + 0.044715 * x**3)))

class FeedForward(nn.Module):
    """
    MLP (emb_dim -> 4 * emb_dim -> emb_dim) implementation.
    We unpacked the layers to cache their outputs for manual optimization.
    """
    def __init__(self, emb_dim):
        super().__init__()
        self.l1 = nn.Linear(emb_dim, 4 * emb_dim)
        self.a1 = GELU()
        self.l2 = nn.Linear(4 * emb_dim, emb_dim)

    def forward(self, x):
        # Normal autograd forward
        z1 = self.l1(x)
        a1 = self.a1(z1)
        z2 = self.l2(a1)
        return z2

We will follow these steps:
- Peel out the layers from the FFN class so we can access the weights and biases directly.
    - $z_1 = X @ W_1 + b_1$
    - $a_1 = GELU(z_1)$
    - $z_2 = a_1 @ W_2 + b_2$
- Store these outputs as cache so we can use them during backpropagation say ffn_cache.
    - Cache: `{ 'X': X, 'z_1': z_1, 'a_1': a_1, 'z_2': z_2 }`
- Write GELU derivative function as we will need it during backpropagation.

    - $GELU(x) = x * \phi(x)$, where $\phi(x)$ is the Cumulative Distribution Function (CDF) for Gaussian Distribution.
    - When the approximation argument is 'tanh', Gelu is estimated with:

        - $GELU(x) = 0.5 * x * (1 + tanh(\sqrt{2/\pi} * (x + 0.044715 * x^3)))$

    - Let 
        - $k = \sqrt{2/\pi}$
        - $u = z_1 + 0.044715 * z_1^3$
        - $t = tanh(k * u)$
    - Therefore, $gelu(z_1) = 0.5 * z_1 * (1 + t)$
    - $dgelu/dz_1 = 0.5 * (1 + t) + 0.5 * z_1 * (1 - t^2) * k * (1 + 3 * 0.044715 * z_1^2)$
    - Cache $k, u, t$ during forward pass for use in backward pass to avoid recomputation.
- Implement the manual backward pass to compute gradients (local jacobians).
    - $p = softmax(z_2)$
    - $dL/dz_2 = p - y\_true$
    - $dL/dW_2 = a_1^T @ dL/dz_2$
    - $dL/db_2 = sum(dL/dz_2)$
    - $dL/da_1 = dL/dz_2 @ W_2^T$
    - $dL/dz_1 = dL/da_1 * dgelu/dz_1$
    - $dL/dW_1 = X^T @ dL/dz_1$
    - $dL/db_1 = sum(dL/dz_1)$
- Update weights and biases using learning rate and computed gradients.
    - $W_2 = W_2 + learning\_rate * (-dL/dW_2)$
    - $b_2 = b_2 + learning\_rate * (-dL/db_2)$
    - $W_1 = W_1 + learning\_rate * (-dL/dW_1)$
    - $b_1 = b_1 + learning\_rate * (-dL/db_1)$
- Use CrossEntropyLoss for loss computation.
    - CE expects Logits over classes. 
- Manual SGD update of weights and biases using the computed gradients.
- Verify the results with autograd.

In [ ]:
@torch.no_grad()
def ffn_forward_with_cache(ffn, X):
    # Extracting layer parameters
    W1 = ffn.l1.weight
    b1 = ffn.l1.bias
    W2 = ffn.l2.weight
    b2 = ffn.l2.bias

    # Calculating forward pass
    z1 = X @ W1.T + b1
    
    # GELU cache parts
    k = torch.sqrt(torch.tensor(2.0 / torch.pi))
    u = z1 + 0.044715 * z1**3
    t = torch.tanh(k * u)

    a1 = 0.5 * z1 * (1 + t)
    z2 = a1 @ W2.T + b2

    # Caching everything
    cache = {
        "X": X,
        "z1": z1,
        "a1": a1,
        "z2": z2,
        "W1": W1,
        "b1": b1,
        "W2": W2,
        "b2": b2,
        "k": k,
        "u": u,
        "t": t
    }
    return z2, cache # We would need z2 for loss calculation


def gelu_backward_tanh(dA1, z1, k, t):
    # du/dz where u = z + 0.044715*z^3, so du/dz = 1 + 3*0.044715*z^2
    du_dz = 1 + 3 * 0.044715 * z1**2
    dgelu_dz = 0.5 * (1 + t) + 0.5 * z1 * (1 - t**2) * k * du_dz
    return dA1 * dgelu_dz

def loss_grad(z2, y_true):
    # softmax i.e. e^z / sum(e^z)
    # stable sofmax: z = z - max(z) # numerical stability
    # and then p = e^z / sum(e^z) # no large values to cause fp-overflow
    z = z2 - z2.max(dim=1, keepdim=True).values
    p = torch.exp(z)
    p = p / p.sum(dim=1, keepdim=True) # after stable softmax

    N, C = p.shape
    y_onehot = torch.zeros_like(p)
    y_onehot[torch.arange(N), y_true] = 1.0
    return (p - y_onehot) / N

def ffn_backward_manual(cache, dL_dz2):
    X, z1, a1 = cache["X"], cache["z1"], cache["a1"]
    W1, W2 = cache["W1"], cache["W2"]
    k, t = cache["k"], cache["t"]

    # Second layer
    dW2 = dL_dz2.T @ a1
    db2 = dL_dz2.sum(dim=0)
    dA1 = dL_dz2 @W2

    # Through GELU
    dZ1 = gelu_backward_tanh(dA1, z1, k, t)

    # First layer
    dW1 = dZ1.T @ X
    db1 = dZ1.sum(dim=0)
    dX = dZ1 @ W1

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads, dX

@torch.no_grad()
def sgd_update(ffn, grads, lr=0.001):
    ffn.l1.weight -= lr * grads["dW1"]
    ffn.l1.bias -= lr * grads["db1"]
    ffn.l2.weight -= lr * grads["dW2"]
    ffn.l2.bias -= lr * grads["db2"]

def train_manual(ffn, X, y, lr=0.001):
    # forward + cache
    z2, cache = ffn_forward_with_cache(ffn, X)
    
    # CE Loss
    with torch.no_grad():
        z = z2 - z2.max(dim=1, keepdim=True).values
        logp = z - torch.logsumexp(z, dim=1, keepdim=True)
        loss = (-logp[torch.arange(X.size(0)), y]).mean().item()

    # backward
    dL_dz2 = loss_grad(z2, y)
    grads, _ = ffn_backward_manual(cache, dL_dz2)

    # update
    sgd_update(ffn, grads, lr=lr)
    return loss


In [ ]:
# --- Quick demo: does our manual backprop actually train? ---
# We make up a tiny classification problem (treating each of the emb_dim output
# columns of z2 as a class) and run our hand-written forward/backward/update loop.
torch.manual_seed(123)

emb_dim = 8
N = 32                                  # number of samples
X = torch.randn(N, emb_dim)
y = torch.randint(0, emb_dim, (N,))     # class label per sample (C = emb_dim columns of z2)

ffn = FeedForward(emb_dim)
for step in range(201):
    loss = train_manual(ffn, X, y, lr=0.1)
    if step % 40 == 0:
        print(f"step {step:3d} | loss {loss:.4f}")

# Loss should decrease steadily -> our manual gradients point downhill.

In [ ]:
# --- Gradient check: our manual gradients vs PyTorch autograd ---
# The ultimate test of the math above: do our hand-derived gradients match
# what autograd computes for the exact same network and loss?
torch.manual_seed(0)
emb_dim = 8
X = torch.randn(4, emb_dim)
y = torch.randint(0, emb_dim, (4,))

ffn = FeedForward(emb_dim)

# Our manual gradients
z2, cache = ffn_forward_with_cache(ffn, X)
dL_dz2 = loss_grad(z2, y)
grads, _ = ffn_backward_manual(cache, dL_dz2)

# Autograd reference (cross_entropy = softmax + negative log likelihood)
ffn.zero_grad()
loss = nn.functional.cross_entropy(ffn(X), y)
loss.backward()

params = dict(ffn.named_parameters())
for name, manual in [("l1.weight", grads["dW1"]), ("l1.bias", grads["db1"]),
                     ("l2.weight", grads["dW2"]), ("l2.bias", grads["db2"])]:
    max_diff = (manual - params[name].grad).abs().max().item()
    print(f"{name:10s} max abs diff vs autograd: {max_diff:.2e}")

# All diffs ~1e-7 (float32 noise) -> our backprop derivation is correct.